In [10]:
import sqlite3
import pandas as pd
from collections import Counter


# 1) Conexión a la base de datos SQLite
DB_PATH = "data/AmITheAsshole.sqlite" 
conn = sqlite3.connect(DB_PATH)

# 2) Carga de tablas
df_submissions = pd.read_sql_query(
    "SELECT submission_id, title, selftext FROM submission;",
    conn
)
df_comments = pd.read_sql_query(
    "SELECT submission_id, message FROM comment;",
    conn
)
conn.close()

# 3) Inferencia de etiqueta desde los mensajes
verdict_keywords = ['NTA', 'YTA', 'ESH', 'NAH', 'INFO']

def extract_verdict(msg: str) -> str:
    msg = msg.upper()
    for word in verdict_keywords:
        # busca la palabra exacta en el split
        if word in msg.split():
            return word
    return None

# Filtramos los comentarios que contienen alguna de las etiquetas
df_comments['label'] = df_comments['message'].apply(extract_verdict)
df_comments = df_comments[df_comments['label'].notnull()]

# 4) Para cada post, quedarnos con la etiqueta más frecuente
submission_verdicts = (
    df_comments
    .groupby('submission_id')['label']
    .apply(lambda x: Counter(x).most_common(1)[0][0])
    .reset_index()
)

# 5) Merge con los submissions y creación de la columna 'text'
df_merged = df_submissions.merge(submission_verdicts, on='submission_id')
df_merged['text'] = (
    df_merged['title'].fillna("") + " " +
    df_merged['selftext'].fillna("")
)

# Se crea dataframe final
df_final = df_merged.rename(columns={'label':'gold_label'})
print(df_final.head())
print(f"Total posts extraídos: {len(df_final)}")

# 6) Guardar el dataset final
df_final.to_csv("aita_final_dataset_2.csv", index=False)

  submission_id                                              title  \
0        xt1ksm            AITA Monthly Open Forum Spooktober 2022   
1        yiplwk  AITA for asking my friend to move a picture of...   
2        yiv572  AITA for asking my husband to stay with me whi...   
3        yimgaf  AITA for telling my SIL to stop talking about ...   
4        yin7pf  AITA for wanting to meet my "daughter" after g...   

                                            selftext gold_label  \
0  #Keep things civil. Rules still apply.\n\n##Th...        NTA   
1  \n\nMe (M32) and my wife, Dahlia (F28) lost ou...        YTA   
2  Throwaway my family knows my account. I'll get...        YTA   
3  My (37M) wife (37F) is pregnant with our first...        NTA   
4  Long story short: in my (40f) twenties I had a...        YTA   

                                                text  
0  AITA Monthly Open Forum Spooktober 2022 #Keep ...  
1  AITA for asking my friend to move a picture of...  
2  AITA for

In [3]:
import numpy as np

def sample_aita_dataset(df: pd.DataFrame, fraction: float = 0.35, random_state: int = 42) -> pd.DataFrame:
    """
    Retorna una muestra aleatoria del DataFrame original con un porcentaje especificado.

    Parámetros:
    - df (pd.DataFrame): El DataFrame original.
    - fraction (float): Proporción de entradas a seleccionar (default 0.35).
    - random_state (int): Semilla para reproducibilidad.

    Retorna:
    - pd.DataFrame: DataFrame con la muestra aleatoria.
    """
    return df.sample(frac=fraction, random_state=random_state).reset_index(drop=True)

# Crear una muestra aleatoria del 35%
df_sampled = sample_aita_dataset(df_final, fraction=0.35)

# Guardar el dataset reducido
df_sampled.to_csv("aita_dataset_35.csv", index=False)

print(f"Posts en el dataset reducido: {len(df_sampled)}")


Posts en el dataset reducido: 10763


In [4]:
def remove_errored_submissions(df: pd.DataFrame, error_csv_path: str) -> pd.DataFrame:
    """
    Elimina entradas del DataFrame original que tienen submission_id en el CSV de errores.

    Parámetros:
    - df (pd.DataFrame): DataFrame original.
    - error_csv_path (str): Ruta al CSV con submission_id erróneos.

    Retorna:
    - pd.DataFrame: DataFrame limpio.
    """
    # Cargar lista de IDs con error
    df_errors = pd.read_csv(error_csv_path)
    
    # Verificar que la columna exista
    if 'submission_id' not in df_errors.columns:
        raise ValueError("El CSV de errores debe tener una columna llamada 'submission_id'")
    
    # Filtrar el dataframe
    df_clean = df[~df['submission_id'].isin(df_errors['submission_id'])]
    
    return df_clean.reset_index(drop=True)


In [ ]:
# Usar la función para limpiar df_final
df_clean = remove_errored_submissions(df_sampled, "aita_errores_filtrados.csv")

# Guardar el resultado limpio
df_clean.to_csv("aita_dataset_limpio_35.csv", index=False)

print(f"Posts después de limpiar errores: {len(df_clean)}")


Posts después de limpiar errores: 10615


## Prueba con 0 de temperatura

In [9]:
from openai import AzureOpenAI
import os
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from dotenv import load_dotenv
from tqdm import tqdm 


# 1) Configuración de AzureOpenAI
load_dotenv()

deployment_name = "gpt"

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT", "https://invuniandesai-2.openai.azure.com/"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-10-21"
)


# 2) Función de clasificación AITA
def clasifica_aita(texto: str, index: int = None) -> str:
    try:
        respuesta = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content":
                 "Eres un clasificador de etiquetas para el subreddit AITA. "
                 "Responde únicamente con UNA de estas etiquetas: YTA, NTA, NAH, ESH o INFO."},
                {"role": "user", "content": texto}
            ],
            temperature=0
        )
        etiqueta = respuesta.choices[0].message.content.strip()
        return etiqueta
    except Exception as e:
        print(f"⚠️ Entrada con error en índice {index}: {e}")
        return "ERROR"


# 3) Procesamiento del DataFrame con tqdm y logging de errores
df = pd.read_csv("aita_dataset_limpio_35.csv")
predicciones = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Clasificando AITA"):
    pred = clasifica_aita(row["text"], index=idx)
    predicciones.append(pred)

df["pred_openai"] = predicciones

# 4) Cálculo de métricas
df_valid = df[df["pred_openai"] != "ERROR"]
y_true = df_valid["gold_label"]
y_pred = df_valid["pred_openai"]
labels = ["YTA", "NTA", "NAH", "ESH", "INFO"]

print("=== Métricas Azure OpenAI ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}\n")
print(classification_report(y_true, y_pred, labels=labels, digits=4))
print("Matriz de confusión:\n", confusion_matrix(y_true, y_pred, labels=labels))

# 5) Guardar resultados
df[["submission_id", "gold_label", "pred_openai"]] \
    .to_csv("aita_openai_results.csv", index=False)

# 6) Reporte de errores
errores = df[df["pred_openai"] == "ERROR"]
print(f"\nTotal de entradas con error: {len(errores)}")
if not errores.empty:
    errores.to_csv("aita_errores_filtrados.csv", index=False)
    print("Errores guardados en 'aita_errores_filtrados.csv'")

Clasificando AITA: 100%|██████████| 10615/10615 [3:13:26<00:00,  1.09s/it]    

=== Métricas Azure OpenAI ===
Accuracy: 62.21%

              precision    recall  f1-score   support

         YTA     0.3263    0.6860    0.4423      1901
         NTA     0.9259    0.6172    0.7407      8548
         NAH     0.0530    0.1290    0.0751        62
         ESH     0.0287    0.1975    0.0502        81
        INFO     0.0000    0.0000    0.0000        23

    accuracy                         0.6221     10615
   macro avg     0.2668    0.3259    0.2616     10615
weighted avg     0.8046    0.6221    0.6765     10615

Matriz de confusión:
 [[1304  371   31  147   48]
 [2625 5276  105  389  153]
 [  15   25    8    4   10]
 [  49   14    0   16    2]
 [   3   12    7    1    0]]

Total de entradas con error: 0


## Temperatura 0.4

In [14]:
from openai import AzureOpenAI
import os
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from dotenv import load_dotenv
from tqdm import tqdm 


# 1) Configuración de AzureOpenAI
load_dotenv()

deployment_name = "gpt"

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT", "https://invuniandesai-2.openai.azure.com/"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-10-21"
)


# 2) Función de clasificación AITA
def clasifica_aita(texto: str, index: int = None) -> str:
    try:
        respuesta = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content":
                 "Eres un clasificador de etiquetas para el subreddit AITA. "
                 "Responde únicamente con UNA de estas etiquetas: YTA, NTA, NAH, ESH o INFO."},
                {"role": "user", "content": texto}
            ],
            temperature=0.4
        )
        etiqueta = respuesta.choices[0].message.content.strip()
        return etiqueta
    except Exception as e:
        print(f"⚠️ Entrada con error en índice {index}: {e}")
        return "ERROR"


# 3) Procesamiento del DataFrame con tqdm y logging de errores
df = pd.read_csv("aita_dataset_limpio_35.csv")
predicciones = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Clasificando AITA"):
    pred = clasifica_aita(row["text"], index=idx)
    predicciones.append(pred)

df["pred_openai"] = predicciones

# 4) Cálculo de métricas
df_valid = df[df["pred_openai"] != "ERROR"]
y_true = df_valid["gold_label"]
y_pred = df_valid["pred_openai"]
labels = ["YTA", "NTA", "NAH", "ESH", "INFO"]

print("=== Métricas Azure OpenAI ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}\n")
print(classification_report(y_true, y_pred, labels=labels, digits=4))
print("Matriz de confusión:\n", confusion_matrix(y_true, y_pred, labels=labels))

# 5) Guardar resultados
df[["submission_id", "gold_label", "pred_openai"]] \
    .to_csv("aita_openai_results_t04.csv", index=False)

# 6) Reporte de errores
errores = df[df["pred_openai"] == "ERROR"]
print(f"\nTotal de entradas con error: {len(errores)}")
if not errores.empty:
    errores.to_csv("aita_errores_filtrados.csv", index=False)
    print("Errores guardados en 'aita_errores_filtrados.csv'")

Clasificando AITA: 100%|██████████| 10615/10615 [1:34:25<00:00,  1.87it/s] 


=== Métricas Azure OpenAI ===
Accuracy: 61.36%

              precision    recall  f1-score   support

         YTA     0.3223    0.6823    0.4378      1901
         NTA     0.9263    0.6074    0.7337      8548
         NAH     0.0485    0.1290    0.0705        62
         ESH     0.0265    0.1975    0.0467        81
        INFO     0.0000    0.0000    0.0000        23

    accuracy                         0.6136     10615
   macro avg     0.2647    0.3232    0.2577     10615
weighted avg     0.8041    0.6136    0.6700     10615

Matriz de confusión:
 [[1297  358   35  159   52]
 [2662 5192  116  424  154]
 [  14   27    8    4    9]
 [  48   15    0   16    2]
 [   3   13    6    1    0]]

Total de entradas con error: 0


## Temperatura 0.8

In [15]:
from openai import AzureOpenAI
import os
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from dotenv import load_dotenv
from tqdm import tqdm 


# 1) Configuración de AzureOpenAI
load_dotenv()

deployment_name = "gpt"

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT", "https://invuniandesai-2.openai.azure.com/"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-10-21"
)


# 2) Función de clasificación AITA
def clasifica_aita(texto: str, index: int = None) -> str:
    try:
        respuesta = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content":
                 "Eres un clasificador de etiquetas para el subreddit AITA. "
                 "Responde únicamente con UNA de estas etiquetas: YTA, NTA, NAH, ESH o INFO."},
                {"role": "user", "content": texto}
            ],
            temperature=0.8
        )
        etiqueta = respuesta.choices[0].message.content.strip()
        return etiqueta
    except Exception as e:
        print(f"⚠️ Entrada con error en índice {index}: {e}")
        return "ERROR"


# 3) Procesamiento del DataFrame con tqdm y logging de errores
df = pd.read_csv("aita_dataset_limpio_35.csv")
predicciones = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Clasificando AITA"):
    pred = clasifica_aita(row["text"], index=idx)
    predicciones.append(pred)

df["pred_openai"] = predicciones

# 4) Cálculo de métricas
df_valid = df[df["pred_openai"] != "ERROR"]
y_true = df_valid["gold_label"]
y_pred = df_valid["pred_openai"]
labels = ["YTA", "NTA", "NAH", "ESH", "INFO"]

print("=== Métricas Azure OpenAI ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}\n")
print(classification_report(y_true, y_pred, labels=labels, digits=4))
print("Matriz de confusión:\n", confusion_matrix(y_true, y_pred, labels=labels))

# 5) Guardar resultados
df[["submission_id", "gold_label", "pred_openai"]] \
    .to_csv("aita_openai_results_t08.csv", index=False)

# 6) Reporte de errores
errores = df[df["pred_openai"] == "ERROR"]
print(f"\nTotal de entradas con error: {len(errores)}")
if not errores.empty:
    errores.to_csv("aita_errores_filtrados.csv", index=False)
    print("Errores guardados en 'aita_errores_filtrados.csv'")

Clasificando AITA: 100%|██████████| 10615/10615 [8:31:29<00:00,  2.89s/it]      

=== Métricas Azure OpenAI ===
Accuracy: 60.64%

              precision    recall  f1-score   support

         YTA     0.3194    0.6718    0.4330      1901
         NTA     0.9272    0.6007    0.7291      8548
         NAH     0.0314    0.1129    0.0491        62
         ESH     0.0275    0.2099    0.0486        81
        INFO     0.0042    0.0435    0.0077        23

   micro avg     0.6065    0.6064    0.6064     10615
   macro avg     0.2619    0.3277    0.2535     10615
weighted avg     0.8043    0.6064    0.6653     10615

Matriz de confusión:
 [[1277  353   49  169   52]
 [2654 5135  160  430  169]
 [  15   25    7    3   12]
 [  49   13    0   17    2]
 [   3   12    7    0    1]]

Total de entradas con error: 0


## Temperatura 1.0

In [16]:
from openai import AzureOpenAI
import os
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from dotenv import load_dotenv
from tqdm import tqdm 


# 1) Configuración de AzureOpenAI
load_dotenv()

deployment_name = "gpt"

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT", "https://invuniandesai-2.openai.azure.com/"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-10-21"
)


# 2) Función de clasificación AITA
def clasifica_aita(texto: str, index: int = None) -> str:
    try:
        respuesta = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content":
                 "Eres un clasificador de etiquetas para el subreddit AITA. "
                 "Responde únicamente con UNA de estas etiquetas: YTA, NTA, NAH, ESH o INFO."},
                {"role": "user", "content": texto}
            ],
            temperature=0.4
        )
        etiqueta = respuesta.choices[0].message.content.strip()
        return etiqueta
    except Exception as e:
        print(f"⚠️ Entrada con error en índice {index}: {e}")
        return "ERROR"


# 3) Procesamiento del DataFrame con tqdm y logging de errores
df = pd.read_csv("aita_dataset_limpio_35.csv")
predicciones = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Clasificando AITA"):
    pred = clasifica_aita(row["text"], index=idx)
    predicciones.append(pred)

df["pred_openai"] = predicciones

# 4) Cálculo de métricas
df_valid = df[df["pred_openai"] != "ERROR"]
y_true = df_valid["gold_label"]
y_pred = df_valid["pred_openai"]
labels = ["YTA", "NTA", "NAH", "ESH", "INFO"]

print("=== Métricas Azure OpenAI ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}\n")
print(classification_report(y_true, y_pred, labels=labels, digits=4))
print("Matriz de confusión:\n", confusion_matrix(y_true, y_pred, labels=labels))

# 5) Guardar resultados
df[["submission_id", "gold_label", "pred_openai"]] \
    .to_csv("aita_openai_results_t10.csv", index=False)

# 6) Reporte de errores
errores = df[df["pred_openai"] == "ERROR"]
print(f"\nTotal de entradas con error: {len(errores)}")
if not errores.empty:
    errores.to_csv("aita_errores_filtrados.csv", index=False)
    print("Errores guardados en 'aita_errores_filtrados.csv'")

Clasificando AITA: 100%|██████████| 10615/10615 [4:25:59<00:00,  1.50s/it]     

=== Métricas Azure OpenAI ===
Accuracy: 61.31%

              precision    recall  f1-score   support

         YTA     0.3204    0.6781    0.4352      1901
         NTA     0.9256    0.6083    0.7342      8548
         NAH     0.0361    0.0968    0.0526        62
         ESH     0.0222    0.1605    0.0390        81
        INFO     0.0000    0.0000    0.0000        23

    accuracy                         0.6131     10615
   macro avg     0.2609    0.3087    0.2522     10615
weighted avg     0.8031    0.6131    0.6697     10615

Matriz de confusión:
 [[1289  362   35  163   52]
 [2666 5200  119  403  160]
 [  13   30    6    4    9]
 [  53   13    0   13    2]
 [   2   13    6    2    0]]

Total de entradas con error: 0
